# DataMate — Kaggle GPU smoke test

Mục tiêu: xác nhận pipeline chạy được với LLM thật trên GPU Kaggle (T4 x2).
Đây **không** phải bài đo 6 giờ; máy chấm là 2 x RTX 5090.

Cài đặt notebook trước khi chạy (Settings bên phải):
- Accelerator: **GPU T4 x2**
- Internet: **On** (chỉ cần cho cell cài đặt / clone; tắt được ở cell suy luận)
- Persistence: Files only
- Input: add dataset chứa trọng số `Qwen2.5-Coder-7B-Instruct-AWQ` (xem cell 2 nếu chưa có)

In [ ]:
# Cell 1 — lấy source code
import os, subprocess, pathlib
REPO_URL = "https://github.com/HoL0ngg/Zalo-AI-Challenge-2026.git"
ROOT = pathlib.Path("/kaggle/working/datamate")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Cell 2 — trọng số mô hình.
# Lần đầu: tải từ Hugging Face (Internet On), rồi Save Version -> Output -> "Create Dataset"
# để các lần sau add làm Input và chạy offline. Đổi WEIGHTS_INPUT theo tên dataset của bạn.
import glob, pathlib
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"
WEIGHTS_INPUT = glob.glob("/kaggle/input/*/Qwen2.5-Coder-7B-Instruct-AWQ")
if WEIGHTS_INPUT:
    weights = pathlib.Path(WEIGHTS_INPUT[0])
    print("Dùng trọng số có sẵn:", weights)
else:
    weights = pathlib.Path("/kaggle/working/Qwen2.5-Coder-7B-Instruct-AWQ")
    if not (weights / "config.json").exists():
        from huggingface_hub import snapshot_download
        snapshot_download(MODEL_ID, local_dir=str(weights))
    print("Đã tải trọng số về:", weights, "— hãy Save Version và tạo Dataset từ Output")

link = ROOT / "models" / "qwen-coder-awq"          # phải khớp local_path trong configs/models_kaggle.yaml
link.parent.mkdir(exist_ok=True)
if link.is_symlink() or link.exists():
    link.unlink()
link.symlink_to(weights, target_is_directory=True)
assert (link / "config.json").exists(), "Symlink trọng số chưa đúng"
print("models/qwen-coder-awq ->", link.resolve())

In [ ]:
# Cell 3 — cài phụ thuộc (5–10 phút). vLLM kéo theo bản torch của riêng nó.
%pip install -q -e ".[test]"
%pip install -q rapidfuzz Unidecode sqlglot
%pip install -q vllm
import vllm, torch
print("vllm", vllm.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# Cell 4 — kiểm tra CPU/mock trước khi đụng GPU
!python -m pytest -q
!python -m datamate.cli run --data-dir data/samples --result-dir /kaggle/working/result/sample
!python -m datamate.cli eval /kaggle/working/result/sample/submission.jsonl data/samples/answers.jsonl

In [ ]:
# Cell 5 — chạy LLM thật trên tập train (có answers.jsonl để chấm). Từ cell này có thể tắt Internet.
import os
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                  VLLM_NO_USAGE_STATS="1", PYTHONHASHSEED="20261030")
!python -m datamate.cli run --config configs/kaggle.yaml     --data-dir dataset/train --result-dir /kaggle/working/result/train

In [ ]:
# Cell 6 — chấm điểm, kiểm tra định dạng, xem vài log
!python -m datamate.cli validate /kaggle/working/result/train/submission.jsonl
!python -m datamate.cli eval /kaggle/working/result/train/submission.jsonl dataset/train/answers.jsonl
!ls /kaggle/working/result/train/logs | head -5
!head -c 1500 /kaggle/working/result/train/submission.jsonl

In [ ]:
# Cell 7 (tuỳ chọn) — chạy lần 2 để kiểm tra tính tái lập (yêu cầu lệch <= 0.03)
!python -m datamate.cli run --config configs/kaggle.yaml     --data-dir dataset/train --result-dir /kaggle/working/result/train_rerun
!python -m datamate.cli eval /kaggle/working/result/train_rerun/submission.jsonl dataset/train/answers.jsonl
!diff <(sort /kaggle/working/result/train/submission.jsonl) <(sort /kaggle/working/result/train_rerun/submission.jsonl) | head -20

In [ ]:
# Cell 8 (tuỳ chọn) — tập dev không có đáp án, chỉ kiểm tra pipeline không crash
!python -m datamate.cli run --config configs/kaggle.yaml     --data-dir dataset/dev --result-dir /kaggle/working/result/dev
!python -m datamate.cli validate /kaggle/working/result/dev/submission.jsonl